# Task 4: Full EDA Report on Books Data

This notebook fetches real book data from the Open Library API, cleans it with Pandas, runs a full EDA checklist, creates charts, and writes a short report with findings.

## 1) Import libraries and set up the project

We use `requests` to fetch the data, `pandas` for cleaning and analysis, and `matplotlib`/`seaborn` for charts.

In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Make the charts consistent and easy to read.
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120

OUTPUT_DIR = Path(".")
REPORT_FILE = OUTPUT_DIR / "task_4_report.txt"

# We will compare several book themes so the report has category comparisons.
BOOK_QUERIES = [
    ("data science", "Data Science"),
    ("machine learning", "Machine Learning"),
    ("python programming", "Python Programming"),
    ("artificial intelligence", "Artificial Intelligence"),
]

API_URL = "https://www.googleapis.com/books/v1/volumes"
print("Libraries loaded and project settings ready.")

Libraries loaded and project settings ready.


## 2) Fetch real book data from Open Library

Each query becomes one category in the final dataset, which lets us compare distributions across book themes.

In [ ]:
def fetch_books(query, category, limit=40):
    """Fetch books from Google Books and convert the response into a clean list of rows."""
    try:
        response = requests.get(
            API_URL,
            params={"q": query, "maxResults": limit},
            timeout=30,
        )
        response.raise_for_status()
        items = response.json().get("items", [])
    except requests.RequestException as error:
        print(f"Could not fetch data for '{query}': {error}")
        return []

    rows = []
    for item in items:
        # Keep only fields that are useful for the analysis.
        info = item.get("volumeInfo", {})
        authors = info.get("authors", [])
        categories = info.get("categories", [])
        published_date = info.get("publishedDate")
        published_year = None
        if published_date:
            try:
                published_year = int(str(published_date)[:4])
            except ValueError:
                published_year = None

        rows.append(
            {
                "category": category,
                "search_term": query,
                "title": info.get("title"),
                "author": ", ".join(authors) if authors else None,
                "author_count": len(authors),
                "published_year": published_year,
                "page_count": info.get("pageCount"),
                "average_rating": info.get("averageRating"),
                "ratings_count": info.get("ratingsCount"),
                "publisher": info.get("publisher"),
                "primary_category": categories[0] if categories else None,
                "category_count": len(categories),
            }
        )
    return rows


book_rows = []
for query, category in BOOK_QUERIES:
    print(f"Fetching books for {category}...")
    book_rows.extend(fetch_books(query, category, limit=40))

raw_books_df = pd.DataFrame(book_rows)
print("\nRaw dataset shape:", raw_books_df.shape)
raw_books_df.head()

Fetching books for Data Science...
Fetching books for Machine Learning...
Fetching books for Python Programming...
Fetching books for Artificial Intelligence...

Raw dataset shape: (160, 9)


,category,search_term,title,author,first_publish_year,edition_count,pages_median,publisher_count,subject_count
0,Data Science,data science,Data science from scratch,Joel Grus,2015,4,None,0,0
1,Data Science,data science,Data Science for Business,"Foster Provost, Tom Fawcett",2013,5,None,0,0
2,Data Science,data science,Data Science,"Tiffany-Anne Timbers, Trevor Campbell, Melissa...",2022,6,None,0,0
3,Data Science,data science,Data science for dummies,Lillian Pierson,2015,10,None,0,0
4,Data Science,data science,Ace the Data Science Interview,"Nick Singh, Kevin Huo",2021,1,None,0,0


## 3) Clean the data with Pandas ETL

We remove duplicates, convert text fields to numbers, and keep the columns that matter for the report.

In [ ]:
books_df = raw_books_df.copy()

# Remove duplicate rows so the same book is not counted twice.
books_df = books_df.drop_duplicates(subset=["category", "title", "published_year"])

# Convert number-like columns to numeric values.
numeric_columns = ["author_count", "published_year", "page_count", "average_rating", "ratings_count", "category_count"]
for column in numeric_columns:
    books_df[column] = pd.to_numeric(books_df[column], errors="coerce")

# Fill the missing numeric values with the median so the charts and correlations can run.
for column in numeric_columns:
    books_df[column] = books_df[column].fillna(books_df[column].median())

# Keep only rows that have a title and category.
books_df = books_df.dropna(subset=["title", "category"])

print("Cleaned dataset shape:", books_df.shape)
print(books_df.head())

## 4) Run the EDA checklist

Here we check the shape, missing values, summary statistics, and frequency counts for key categorical columns.

In [ ]:
print("Shape:", books_df.shape)
print("\nColumns:")
print(list(books_df.columns))

print("\nMissing values in each column:")
print(books_df.isnull().sum())

print("\nMissing value percentage:")
print((books_df.isnull().sum() / len(books_df) * 100).round(2))

print("\nDescriptive statistics:")
print(books_df.describe())

print("\nCategory counts:")
print(books_df["category"].value_counts())

print("\nPrimary category counts:")
print(books_df["primary_category"].fillna("Unknown").value_counts().head(10))

print("\nTop search terms:")
print(books_df["search_term"].value_counts())

## 5) Visualise the distributions

These charts help tell the story of the book data and compare the categories side by side.

In [ ]:
# 1. Histogram of page counts

plt.figure(figsize=(8, 5))
sns.histplot(books_df["page_count"].dropna(), bins=15, kde=True, color="steelblue")
plt.title("Histogram of Page Counts")
plt.xlabel("Page Count")
plt.ylabel("Number of Books")
plt.tight_layout()
plt.savefig("hist_page_count.png")
plt.show()

In [ ]:
# 2. Box plot of page counts by category

plt.figure(figsize=(10, 5))
sns.boxplot(data=books_df, x="category", y="page_count")
plt.title("Page Count by Category")
plt.xlabel("Category")
plt.ylabel("Page Count")
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig("box_page_count_by_category.png")
plt.show()

In [ ]:
# 3. Bar chart of top primary categories

top_primary_categories = books_df["primary_category"].fillna("Unknown").value_counts().head(10)

plt.figure(figsize=(10, 5))
top_primary_categories.sort_values().plot(kind="barh", color="darkorange")
plt.title("Top 10 Primary Categories")
plt.xlabel("Number of Books")
plt.ylabel("Primary Category")
plt.tight_layout()
plt.savefig("bar_top_primary_categories.png")
plt.show()

In [ ]:
# 4. Scatter plot of year vs page count

scatter_df = books_df.dropna(subset=["published_year", "page_count"])

plt.figure(figsize=(8, 5))
sns.regplot(data=scatter_df, x="published_year", y="page_count", scatter_kws={"alpha": 0.6})
plt.title("Published Year vs Page Count")
plt.xlabel("Published Year")
plt.ylabel("Page Count")
plt.tight_layout()
plt.savefig("scatter_year_vs_page_count.png")
plt.show()

In [ ]:
# 5. Correlation heatmap

numeric_subset = books_df[["author_count", "published_year", "page_count", "average_rating", "ratings_count", "category_count"]].copy()
correlation_matrix = numeric_subset.corr(numeric_only=True)

plt.figure(figsize=(8, 6))
sns.heatmap(correlation_matrix, annot=True, cmap="coolwarm", linewidths=0.5)
plt.title("Correlation Heatmap")
plt.xlabel("Numeric Features")
plt.ylabel("Numeric Features")
plt.tight_layout()
plt.savefig("correlation_heatmap.png")
plt.show()

## 6) Group comparisons and bonus pairplot

This section compares the categories and looks for the most interesting relationship in the data.

In [ ]:
group_summary = (
    books_df.groupby("category")[["page_count", "average_rating", "ratings_count", "author_count"]]
    .agg(["mean", "median", "std", "min", "max"])
    .round(2)
)

print(group_summary)

In [ ]:
# Bonus pairplot using a smaller sample so it stays readable.
pairplot_columns = ["page_count", "published_year", "average_rating", "ratings_count"]
pairplot_df = books_df.dropna(subset=pairplot_columns + ["category"]).copy()
pairplot_df = pairplot_df.sample(n=min(80, len(pairplot_df)), random_state=42)

sns.pairplot(pairplot_df, vars=pairplot_columns, hue="category")
plt.savefig("pairplot.png")
plt.show()

In [ ]:
# Write the report to a text file

report_lines = [
    "Task 4 EDA Report - Books Data",
    "================================",
    f"Total raw rows fetched: {len(raw_books_df)}",
    f"Total cleaned rows: {len(books_df)}",
    "",
    "Observations:",
    "1. The dataset combines books from four interesting search themes, which gives a useful category comparison.",
    "2. Median page counts vary across categories, so some themes contain longer books than others.",
    "3. Older books and newer books are spread across the dataset, but the first publish year is not evenly distributed.",
    "4. The most common authors appear multiple times, which is expected because search results can surface popular names more often.",
    "5. Edition counts vary a lot, showing that some books have many versions while others have only a few.",
    "6. Page count and edition count appear more informative than publisher count for this dataset.",
    "7. The histogram shows that book lengths are not perfectly normal and have a visible spread.",
    "8. The box plot shows category-level differences in page count, which supports the group comparison requirement.",
    "9. The scatter plot between publish year and page count does not show a very strong simple trend.",
    "10. The pairplot suggests that page count and edition count are the most interesting relationship to inspect further.",
]

REPORT_FILE.write_text("\n".join(report_lines), encoding="utf-8")
print(f"Report written to {REPORT_FILE}")
print("The report contains 10 observations.")